# LemGendary Master Execution: AiHelperLemgend (v16.9.240 Nuclear-Hardened Colab Edition)
This unified notebook handles environment synchronization and automated cloud training.


## 1. Hardware Sentinel
Ensure the manifold has the required hardware acceleration.


In [ ]:
import os
import subprocess
import sys
import time

# Runtime environment (LemGendary env-manager SSOT)
os.environ['FOR_DISABLE_CONSOLE_CTRL_HANDLER'] = '1'
os.environ['FOR_IGNORE_EXCEPTIONS'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONUTF8'] = '1'
print('[OK] [SENTINEL] Auditing Hardware Manifold...')
print('[OK] [RECOMMENDED ACCELERATOR] Google Colab: T4 GPU (or A100/L4 with Pro)')

# torch is probed in a fresh interpreter, never imported into this kernel process.
HW_PROBE = '''
import os
import sys
import torch
tpu_detected = bool(
    os.environ.get('TPU_NAME') or os.environ.get('KAGGLE_TPU_NAME')
    or os.path.exists('/dev/vfio/0')
)
xla_devices = []
if not torch.cuda.is_available():
    try:
        os.environ.setdefault('PJRT_DEVICE', 'TPU')
        import torch_xla.core.xla_model as xm
        xla_devices = xm.get_xla_supported_devices()
        tpu_detected = bool(xla_devices)
    except Exception:
        pass
if xla_devices:
    print(f'[OK] [HARDWARE] Google Cloud TPU XLA validated '
          f'({len(xla_devices)} cores).')
    sys.exit(0)
if tpu_detected:
    print('[OK] [HARDWARE] TPU VM detected; torch_xla will be initialized in deps.')
    sys.exit(0)
if not torch.cuda.is_available():
    print('[CRITICAL ERROR] [HARDWARE] NO GPU OR TPU DETECTED! '
          'Training cannot proceed on CPU.')
    print('[ACTION REQUIRED] Enable GPU or TPU Accelerator before '
          'running this notebook:')
    print('   -> Colab:  Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU')
    sys.exit(3)
gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
archs = torch.cuda.get_arch_list()
compat = any(f'{cap[0]}.{cap[1]}' in a or f'sm_{cap[0]}{cap[1]}' in a for a in archs)
try:
    probe = torch.ones(1, device='cuda') + 1.0
    torch.cuda.synchronize()
    del probe
except RuntimeError as kernel_err:
    print(f'[WARNING] [HARDWARE] CUDA kernel test failed: {kernel_err}')
    compat = False
if compat:
    print(f'[OK] [HARDWARE] NVIDIA {gpu_name} (sm_{cap[0]}{cap[1]}) '
          'validated & ready.')
else:
    print(f'[CRITICAL ERROR] [HARDWARE] NVIDIA {gpu_name} '
          f'(sm_{cap[0]}{cap[1]}) has no '
          'kernel images in the preinstalled PyTorch build.')
    print('[ACTION REQUIRED] Switch Colab Runtime to T4 GPU (Runtime -> Change runtime type -> T4 GPU).')
    print('[INFO] The dependency cell installs the suite torch build for this GPU.')
props = torch.cuda.get_device_properties(0)
tag = '[OK] [ACTIVE]' if compat else '[WARNING] [INCOMPATIBLE ACCELERATOR]'
print(f'{tag} {props.name} (Compute Capability sm_{cap[0]}{cap[1]})')
print(f'[OK] [VRAM] {props.total_memory / 1024**3:.1f} GB')
if torch.cuda.device_count() > 1:
    print(f'[OK] [MULTI-GPU] Detected {torch.cuda.device_count()} active GPUs.')
if props.total_memory / 1024**3 < 10.0:
    print('[WARNING] Low VRAM detected. '
          'Suite will enable Survival Profiles automatically.')
'''
hw_probe = subprocess.run([sys.executable, '-c', HW_PROBE],
                          check=False, capture_output=True, text=True)
print(hw_probe.stdout, end='')
if hw_probe.returncode == 3:
    raise RuntimeError('[ABORT] No GPU accelerator detected. Enable GPU in Colab Runtime settings and re-run from the top.')
if hw_probe.returncode != 0:
    print(hw_probe.stderr[-2000:])
    raise RuntimeError(f'[ABORT] Hardware probe failed with exit code {hw_probe.returncode}.')


## 2. Cloud Auth & Secrets


In [ ]:
import os

from google.colab import userdata

SECRET_NAMES = ('SUITE_PAT', 'KAGGLE_API_TOKEN')

def read_secret(name):
    try:
        return userdata.get(name) or None
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
        print(f'[SECRETS] {name} not available ({type(exc).__name__}).')
        return None

secrets = {name: read_secret(name) for name in SECRET_NAMES}
for name in ('SUITE_PAT',):
    if secrets[name]:
        os.environ[name] = secrets[name]

kaggle_token = secrets['KAGGLE_API_TOKEN']
if kaggle_token and kaggle_token.startswith('KGAT_'):
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    os.environ.pop('KAGGLE_KEY', None)
elif kaggle_token:
    print('[REMEDY] KAGGLE_API_TOKEN must be a Kaggle API token '
          '(KGAT_...); legacy keys are not used.')
else:
    print('[REMEDY] Add the secret KAGGLE_API_TOKEN (your Kaggle API '
          'token, KGAT_...) in the Colab Secrets panel (key icon).')

active = [name for name, value in secrets.items() if value]
print(f'[AUTH] Secrets available: {active}')
if not any(secrets[name] for name in ('SUITE_PAT',)):
    print('[REMEDY] Add SUITE_PAT in the Colab Secrets panel (key icon); private '
          'repositories cannot be cloned without it (legacy GITHUB_PAT is unsupported).')


## 3. Repository Synchronization


In [ ]:
import os
import shutil
import stat
import subprocess
import sys
import tempfile

suite_url = 'https://github.com/lemgenda/lemgendary-training-suite.git'
suite_ref = 'main'
checkout_dir = '/content'
suite_path = '/content/lemgendary-training-suite'
os.makedirs(checkout_dir, exist_ok=True)
# The suite runs from the checkout; outputs and the hub stay in the working dir.
os.environ['LEMGENDARY_WORKSPACE_ROOT'] = '/content'
os.environ['LEMGENDARY_ECOSYSTEM_FILE'] = '/content/lemgendary-training-suite/training/ecosystem.json'

token = os.environ.get('SUITE_PAT') or ''
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass_dir = tempfile.mkdtemp(prefix='lemgendary-askpass-')
if token:
    askpass = os.path.join(askpass_dir, 'askpass.sh')
    with open(askpass, 'w', encoding='utf-8') as fh:
        fh.write('#!/bin/sh\ncase "$1" in Username*) echo x-access-token ;; *) echo "$LEMGENDARY_GIT_TOKEN" ;; esac\n')
    os.chmod(askpass, stat.S_IRWXU)
    git_env['GIT_ASKPASS'] = askpass
    git_env['LEMGENDARY_GIT_TOKEN'] = token
    print('[AUTH] GitHub token supplied to git through GIT_ASKPASS (never stored).')
else:
    print('[WARNING] No SUITE_PAT secret found; trying a public clone.')

def _git(args, cwd=None):
    return subprocess.run(['git', *args], cwd=cwd, env=git_env, check=False, capture_output=True, text=True)

def _sync_repo(url, path, marker, ref):
    valid = os.path.isdir(os.path.join(path, '.git')) and os.path.exists(
        os.path.join(path, marker))
    if not valid:
        if os.path.exists(path):
            shutil.rmtree(path)
        init = _git(['init', '--quiet', path])
        if init.returncode != 0:
            return f'init failed: {init.stderr.strip()}'
        add = _git(['remote', 'add', 'origin', url], cwd=path)
        if add.returncode != 0:
            return f'remote add failed: {add.stderr.strip()}'
    # Reset the origin to the plain URL (older notebooks stored the token in it).
    seturl = _git(['remote', 'set-url', 'origin', url], cwd=path)
    if seturl.returncode != 0:
        return f'remote set-url failed: {seturl.stderr.strip()}'
    # One ref (branch, tag or full commit SHA), fetched shallow and checked out.
    fetch = _git(['fetch', '--depth', '1', 'origin', ref], cwd=path)
    err = None
    if fetch.returncode != 0:
        err = f'fetch of {ref} failed: {fetch.stderr.strip()}'
    else:
        reset = _git(['reset', '--hard', 'FETCH_HEAD'], cwd=path)
        if reset.returncode != 0:
            err = f'reset to {ref} failed: {reset.stderr.strip()}'
        else:
            head = _git(['rev-parse', '--short', 'HEAD'], cwd=path)
            if head.returncode != 0:
                err = f'rev-parse after {ref} failed: {head.stderr.strip()}'
            else:
                print(f'[GIT] {url} at {ref} ({head.stdout.strip()})')
    return err

try:
    suite_error = _sync_repo(suite_url, suite_path, os.path.join('training', 'train.py'), suite_ref)
finally:
    try:
        shutil.rmtree(askpass_dir)
    except OSError as rm_err:
        print(f'[WARNING] Could not remove {askpass_dir}: {rm_err}')

if suite_error:
    print(f'[ERROR] Suite {suite_error}')
    print('[ACTION REQUIRED] Add SUITE_PAT to Colab Secrets with GitHub read permissions.')
    sys.exit(1)
print(f'[OK] Suite ready at {suite_path}')


## 4. Environment & Dependency Installation


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
import tempfile

print('[ENV] Probing hardware accelerator...')

torch_index = 'https://download.pytorch.org/whl/cpu'
accel_type = 'cpu'
if (os.environ.get('TPU_NAME') or os.environ.get('KAGGLE_TPU_NAME')
        or os.path.exists('/dev/vfio/0')):
    accel_type = 'tpu'
    os.environ.setdefault('PJRT_DEVICE', 'TPU')
    os.environ.setdefault('XLA_USE_BF16', '1')
    print('[ENV] TPU VM detected; preserving preinstalled torch/torch_xla stack.')
elif shutil.which('nvidia-smi'):
    torch_index = 'https://download.pytorch.org/whl/cu126'
    accel_type = 'cuda_cu126'
elif shutil.which('rocm-smi'):
    torch_index = 'https://download.pytorch.org/whl/rocm6.1'
    accel_type = 'rocm6.1'
else:
    # A preinstalled torch is asked in a fresh interpreter, never imported here.
    torch_probe = subprocess.run(
        [sys.executable, '-c',
         'import torch; print(torch.version.cuda if torch.cuda.is_available() else ""); '
         'print(getattr(torch.version, "hip", None) or "")'],
        check=False, capture_output=True, text=True)
    if torch_probe.returncode != 0:
        print('[ENV] torch is not installed yet; using the CPU index.')
    else:
        probed = [v.strip() for v in torch_probe.stdout.splitlines()] + ['', '']
        cuda_v, hip_v = probed[0], probed[1]
        if cuda_v:
            cuda_tag = 'cu' + ''.join(cuda_v.split('.')[:2])
            torch_index = f'https://download.pytorch.org/whl/{cuda_tag}'
            accel_type = f'cuda_{cuda_tag}'
        elif hip_v:
            rocm_tag = 'rocm' + '.'.join(hip_v.split('.')[:2])
            torch_index = f'https://download.pytorch.org/whl/{rocm_tag}'
            accel_type = f'rocm_{rocm_tag}'

print(f'[ENV] Accelerator: {accel_type}')
print(f'[ENV] PyTorch index: {torch_index}')

req_candidates = [
    '/content/lemgendary-training-suite/requirements.txt',
    '/content/lemgendary-env-manager/requirements/requirements-training.txt',
]
req_path = next((p for p in req_candidates if os.path.exists(p)), None)

torch_family = ('torch', 'torchvision', 'torchaudio')
torch_pins = (
    ['torch==2.14.1+cu126', 'torchvision==0.29.1+cu126', 'torchaudio==2.11.0+cu126']
    if torch_index == 'https://download.pytorch.org/whl/cu126' else []
)
if req_path:
    print(f'[ENV] Manifest: {req_path}')
    install_req, constraint_args = req_path, []
    if torch_pins:
        # The torch family comes from the build's own index only; the manifest's torch
        # pins name the host build, so they are dropped and the rest is constrained.
        print(f'[ENV] Torch build: {torch_pins}')
        torch_res = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--index-url', torch_index,
             *torch_pins], check=False, capture_output=True, text=True)
        if torch_res.returncode != 0:
            print('[WARNING] Torch build installation finished with non-zero exit code.')
            print(torch_res.stderr[-2000:])
        pin_dir = tempfile.mkdtemp(prefix='lemgendary-pins-')
        install_req = os.path.join(pin_dir, 'requirements-without-torch.txt')
        constraints = os.path.join(pin_dir, 'torch-constraints.txt')
        with open(req_path, encoding='utf-8') as src:
            req_lines = src.readlines()
        with open(install_req, 'w', encoding='utf-8') as dst:
            for line in req_lines:
                name = line.split('#')[0].strip()
                for sep in '=<>!~;[ ':
                    name = name.split(sep)[0]
                if name.lower() not in torch_family:
                    dst.write(line)
        with open(constraints, 'w', encoding='utf-8') as fh:
            fh.write(''.join(pin + chr(10) for pin in torch_pins))
        constraint_args = ['-c', constraints]
    res = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         '--extra-index-url', torch_index,
         '--upgrade-strategy', 'only-if-needed',
         *constraint_args, '-r', install_req],
        check=False, capture_output=True, text=True)
    if res.returncode == 0:
        print('[OK] Environment Ready.')
        # The installed torch is asked in a fresh interpreter: a torch module imported
        # in this kernel process before the install would report the old build.
        version_probe = subprocess.run(
            [sys.executable, '-c',
             'import torch; print(torch.__version__, "on", '
             '"CUDA " + torch.version.cuda if torch.cuda.is_available() else "CPU")'],
            check=False, capture_output=True, text=True)
        if version_probe.returncode == 0:
            print(f'[OK] torch {version_probe.stdout.strip()}')
        else:
            print(f'[WARNING] Installed torch could not be probed (exit code '
                  f'{version_probe.returncode}): {version_probe.stderr[-2000:]}')
    else:
        print('[WARNING] Dependency installation finished with non-zero exit code.')
        if res.stderr:
            print(res.stderr[-2000:])
else:
    print('[ERROR] Could not open requirements file: No such file or directory')
    print("[REMEDY] Ensure 'requirements.txt' exists in the root of the repository.")
    print('[ACTION REQUIRED] Suite clone failed in Step 3 because SUITE_PAT is missing from Colab Secrets.')
    print('[ACTION REQUIRED] Fix: '
          'Go to Colab Secrets and add SUITE_PAT with your GitHub token.')
    print('[INFO] Installing core container streaming dependencies as fallback...')
    fallback_deps = [
        'webdataset==1.0.2', 'litdata>=0.2.0,<1.0.0', 'pyarrow>=25.0.1', 'zstandard>=0.23.0'
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *fallback_deps], check=False)


## 5. SOTA Hub Synchronization (Pull)


In [ ]:
import os

hub_root = '/content/LemGendaryModels'
model_key = 'ai_helper_lemgend'
model_dir = os.path.join(hub_root, model_key)
ckpt_dir = os.path.join(model_dir, 'checkpoints')
nb_dir = os.path.join(model_dir, 'notebooks')
vis_dir = os.path.join(model_dir, 'visualizations')
dist_dir = os.path.join(model_dir, 'dist', 'kaggle')

print(f'[HUB] Initializing Canonical Manifold for {model_key}...')
os.makedirs(ckpt_dir, exist_ok=True)
os.makedirs(nb_dir, exist_ok=True)
os.makedirs(vis_dir, exist_ok=True)
os.makedirs(os.path.join(dist_dir, 'pytorch'), exist_ok=True)
os.makedirs(os.path.join(dist_dir, 'onnx_fp32'), exist_ok=True)
os.makedirs(os.path.join(dist_dir, 'onnx_fp16'), exist_ok=True)
print(f'[OK] Canonical manifold structure ready at {model_dir}')


## 6. Google Drive Mount
Mount Google Drive FUSE for streaming datasets directly.


In [ ]:
import os
print('[MOUNT] Attaching Google Drive FUSE...')
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted successfully. Datasets will be streamed directly from Drive.')


## 7. Kaggle Dataset Acquisition & Manifold Resolution


In [ ]:
import os
import re
import shutil
import subprocess
import sys
model_key = 'ai_helper_lemgend'
kaggle_repo = 'lemtreursi/lemgendizedaihelperlemgend'
primary_manifold = 'LemGendizedAIHelperLemGend'
target_dir = '/content/LemGendaryDatasets'
os.makedirs(target_dir, exist_ok=True)
dest_path = os.path.join(target_dir, primary_manifold)

print(f'[DATA] Resolving dataset manifold for {model_key}...')
print(f'[DATA] Target manifold: {primary_manifold} | Kaggle source: {kaggle_repo}')

def _scan_kaggle_inputs(root='/kaggle/input', max_depth=6):
    if not os.path.isdir(root):
        return []
    results = []
    seen = set()
    _MANIFOLD_SUBDIRS = {'images', 'targets', 'masks', 'forex', 'shards', 'train', 'val', 'test', 'splits'}
    _MANIFOLD_META = {'dataset_info.yaml', 'category.txt', 'classes.txt', 'README.md'}
    def _looks_like_manifold(p):
        if not os.path.isdir(p):
            return False
        try:
            names = os.listdir(p)
        except OSError:
            return False
        if any(f.endswith('.parquet') or f.endswith('.data') or f.endswith('.tar') for f in names):
            return True
        if any(f in names for f in _MANIFOLD_META):
            return True
        for n in names:
            full = os.path.join(p, n)
            if os.path.isdir(full) and n.lower() in _MANIFOLD_SUBDIRS:
                return True
        return False
    def _walk(start, depth=0):
        if depth > max_depth or not os.path.isdir(start):
            return
        if _looks_like_manifold(start):
            real = os.path.realpath(start)
            if real not in seen:
                seen.add(real)
                results.append(start)
            return
        try:
            children = os.listdir(start)
        except OSError:
            return
        for child in children:
            _walk(os.path.join(start, child), depth + 1)
    for top in os.listdir(root):
        top_path = os.path.join(root, top)
        if not os.path.isdir(top_path):
            continue
        if top == 'datasets':
            try:
                owners = os.listdir(top_path)
            except OSError:
                continue
            for owner in owners:
                owner_path = os.path.join(top_path, owner)
                if not os.path.isdir(owner_path):
                    continue
                try:
                    slugs = os.listdir(owner_path)
                except OSError:
                    continue
                for slug in slugs:
                    slug_path = os.path.join(owner_path, slug)
                    if os.path.isdir(slug_path):
                        _walk(slug_path, depth=0)
        elif top == 'models':
            continue
        else:
            _walk(top_path, depth=0)
    return results

_kaggle_inputs = _scan_kaggle_inputs()
if _kaggle_inputs:
    print(f'[PRE-FLIGHT] Discovered {len(_kaggle_inputs)} attached manifold(s) in /kaggle/input.')
    for cand in _kaggle_inputs:
        bname = os.path.basename(cand)
        aliases = {bname, bname.lower()}
        stripped = re.sub(r'^LemGendized', '', bname, flags=re.IGNORECASE)
        if stripped != bname:
            aliases.add(stripped)
            aliases.add(stripped.lower())
            for suf in ('Large', 'Medium', 'Small'):
                if stripped.endswith(suf):
                    stem = stripped[:-len(suf)]
                    aliases.add(stem)
                    aliases.add(stem.lower())
        for alias in aliases:
            if not alias: continue
            dst = os.path.join(target_dir, alias)
            if os.path.exists(dst): continue
            try:
                os.symlink(cand, dst)
                print(f'   -> [OK] [PRE-FLIGHT] {alias} -> {cand}')
            except OSError as e:
                print(f'   -> [WARN] Symlink skipped for {alias}: {e}')

is_ready = os.path.exists(dest_path) and (
    os.path.exists(os.path.join(dest_path, 'images')) or
    os.path.exists(os.path.join(dest_path, 'targets')) or
    os.path.exists(os.path.join(dest_path, 'shards')) or
    any(f.endswith('.parquet') or f.endswith('.data') or f.endswith('.tar') or f.endswith('.csv') or f.endswith('.json') for f in os.listdir(dest_path))
)

if is_ready:
    print(f'[OK] [DATA] Manifold already available at {dest_path} — download skipped.')

if not is_ready:
    os.makedirs(dest_path, exist_ok=True)
    download_ok = False
    _no_download = False
    if os.path.exists('/content/drive/MyDrive'):
        print('[FALLBACK] Checking Google Drive for manifold...')
        drive_cands = [
            f'/content/drive/MyDrive/LemGendaryDatasets/{primary_manifold}',
            f'/content/drive/MyDrive/{primary_manifold}',
            f'/content/drive/MyDrive/LemGendaryDatasets/{primary_manifold.replace("Large", "")}',
        ]
        for cand in drive_cands:
            if os.path.exists(cand):
                try:
                    os.symlink(cand, dest_path)
                    print(f'[OK] Symlinked from Google Drive: {cand} -> {dest_path}')
                    download_ok = True
                    break
                except OSError as link_err:
                    print(f'[WARNING] Drive symlink failed for {cand}: {link_err}')

    if not download_ok and kaggle_repo and not _no_download:
        try:
            print(f'[KAGGLE CLI] Downloading {kaggle_repo} into {dest_path}...')
            res = subprocess.run(
                ['kaggle', 'datasets', 'download', '-d', kaggle_repo, '-p', dest_path, '--unzip'],
                capture_output=True, text=True
            )
            if res.returncode == 0:
                print(f'[OK] [KAGGLE CLI] Successfully downloaded and extracted {kaggle_repo}')
                download_ok = True
            else:
                print(f'[WARNING] Kaggle CLI download returned code {res.returncode}: {res.stderr.strip()[:200]}')
        except Exception as e:
            print(f'[WARNING] Kaggle CLI download error: {e}')
        
        if not download_ok:
            try:
                print('[KAGGLEHUB] Attempting download via kagglehub...')
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)
                import kagglehub
                hub_path = kagglehub.dataset_download(kaggle_repo)
                print(f'[OK] [KAGGLEHUB] Downloaded to {hub_path}')
                for item in os.listdir(hub_path):
                    s = os.path.join(hub_path, item)
                    d = os.path.join(dest_path, item)
                    if not os.path.exists(d):
                        try: os.symlink(s, d)
                        except OSError as link_err:
                            print(f'[INFO] Symlink failed, copying: {link_err}')
                            if os.path.isdir(s): shutil.copytree(s, d)
                            else: shutil.copy2(s, d)
                download_ok = True
            except Exception as e:
                print(f'[WARNING] kagglehub download error: {e}')
    
    if not download_ok and _no_download:
        raise RuntimeError(
            f'[ABORT] Dataset not found in /kaggle/input or /content/drive, '
            f'and notebook_no_download=True. Attach the dataset (Kaggle: sidebar; '
            f'Colab: mount Google Drive) and re-run from the top.'
        )

if os.path.exists(dest_path):
    print(f'[OK] Manifold materialized at: {dest_path}')
    aliases = [primary_manifold.lower(), model_key.lower(), model_key.replace('_', '-'), model_key.replace('_', '')]
    if False:
        aliases.extend(['forex', 'lemgendizedforexuniverse', 'lemgendizedforexuniverselarge'])
    for alias in set(aliases):
        a_path = os.path.join(target_dir, alias)
        if not os.path.exists(a_path):
            try: os.symlink(dest_path, a_path)
            except OSError as link_err:
                print(f'[REMEDY] Alias symlink error for {a_path}: {link_err}')
    if False:
        forex_composite_dir = os.path.join(target_dir, 'LemGendizedForexUniverse')
        os.makedirs(forex_composite_dir, exist_ok=True)
        for item in os.listdir(dest_path):
            if item.startswith('ForexUniverse20') and item.endswith('.parquet'):
                flat = os.path.join(target_dir, item)
                if not os.path.exists(flat):
                    try: os.symlink(os.path.join(dest_path, item), flat)
                    except OSError as link_err:
                        print(f'[REMEDY] Symlink error for {flat}: {link_err}')
        link_alias = os.path.join(target_dir, 'forex')
        if not os.path.exists(link_alias):
            try: os.symlink(dest_path, link_alias)
            except OSError as link_err:
                print(f'[REMEDY] Symlink error for {link_alias}: {link_err}')
else:
    print(f'[ERROR] Could not resolve dataset manifold for {model_key}!')


## 8. Checkpoint & Metric Recovery


In [ ]:
import json
import os
import shutil
import subprocess
import sys

suite_path = '/content/lemgendary-training-suite'

hub_root = '/content/LemGendaryModels'
model_key = 'ai_helper_lemgend'
model_slug = 'ai-helper-lemgend'
model_hub_dir = '/content/LemGendaryModels/ai_helper_lemgend'
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')
os.makedirs(ckpt_hub_dir, exist_ok=True)
print(f'[RECOVERY] Looking for {model_key} checkpoints ({model_slug})...')

def _target_name(fname):
    # latest/last, best, progress and vault_* checkpoints; anything else is not resumed.
    stem, ext = os.path.splitext(fname)
    low = stem.lower()
    if ext.lower() not in ('.pth', '.pt'):
        return None
    if low.startswith('vault_'):
        return fname
    for key, name in (('progress', 'progress'), ('best', 'best'), ('latest', 'latest'),
                      ('last', 'latest')):
        if key in low:
            return name + ext.lower()
    return None

def _model_files(root):
    found = []
    for dirpath, _, files in os.walk(root):
        found += [os.path.join(dirpath, f) for f in files if _target_name(f)]
    return sorted(found)

def _model_dirs(root, max_depth=7):
    # Folders named after the Kaggle slug (hyphenated) or the model key.
    names = {model_slug.lower(), model_key.lower()}
    hits, queue = [], [(root, 0)]
    while queue:
        curr, depth = queue.pop(0)
        try:
            entries = sorted(os.scandir(curr), key=lambda e: e.name)
        except OSError as scan_err:
            print(f'   -> [SKIP] {curr}: {scan_err}')
            continue
        for entry in entries:
            if not entry.is_dir():
                continue
            if entry.name.lower() in names:
                hits.append(entry.path)
            elif depth < max_depth:
                queue.append((entry.path, depth + 1))
    return hits

def _stored_epochs(paths):
    # The suite reads the epochs in a fresh interpreter: torch in this kernel process may
    # be older than the build the dependency cell installed.
    if not paths:
        return {}
    probe = subprocess.run([sys.executable, '-m', 'training.utils.epoch_probe', *paths],
                           cwd=suite_path, check=False, capture_output=True, text=True)
    if probe.stderr.strip():
        print(probe.stderr.rstrip()[-2000:])
    if probe.returncode != 0 or not probe.stdout.strip():
        raise RuntimeError(f'[RECOVERY] Epoch probe failed with exit code '
                           f'{probe.returncode}')
    return json.loads(probe.stdout.strip().splitlines()[-1])

candidates = []
adopted = {}
def _adopt_candidates():
    # Keep the checkpoint with the newest stored epoch; never compare file sizes.
    names = [_target_name(os.path.basename(src)) for src, _ in candidates]
    held = {os.path.join(ckpt_hub_dir, n) for n in names}
    probed = {src for src, _ in candidates} | {p for p in held if os.path.exists(p)}
    epochs = _stored_epochs(sorted(probed))
    for (src, label), name in zip(candidates, names, strict=False):
        dst = os.path.join(ckpt_hub_dir, name)
        src_epoch = epochs.get(src)
        if src_epoch is None:
            print(f'   -> [SKIP] {src}: no readable epoch')
            continue
        dst_epoch = epochs.get(dst) if os.path.exists(dst) else None
        if dst_epoch is not None and src_epoch <= dst_epoch:
            continue
        shutil.copy2(src, dst)
        epochs[dst] = src_epoch
        adopted[name] = (src_epoch, os.path.dirname(src))
        print(f'   -> [OK] {label}: {os.path.basename(src)} (epoch {src_epoch}) -> {name}')

drive_root = '/content/drive/MyDrive'
if os.path.isdir(drive_root):
    for folder in _model_dirs(drive_root):
        for src in _model_files(folder):
            candidates.append((src, 'Google Drive'))

_adopt_candidates()
latest = adopted.get('latest.pth') or adopted.get('latest.pt')
if latest:
    src_dir = latest[1]
    for d in (os.path.dirname(src_dir), src_dir):
        m_path = os.path.join(d, 'metrics.csv')
        if os.path.exists(m_path):
            shutil.copy2(m_path, os.path.join(model_hub_dir, 'metrics.csv'))
            print(f'[METRICS] metrics.csv taken with the epoch {latest[0]} checkpoint')
            break
if not adopted:
    print('   -> [SKIP] No newer checkpoint found; keeping the local hub as it is.')


## 9. Continuous Drive Synchronization


In [ ]:
import os
import shutil
import threading
import time

model_key = 'ai_helper_lemgend'
hub_root = '/content/LemGendaryModels'
model_hub_dir = os.path.join(hub_root, model_key)
ckpt_hub_dir = os.path.join(model_hub_dir, 'checkpoints')

try:
    _found = found_ckpts
except NameError:
    _found = []

drive_target_dir = None
if _found:
    drive_target_dir = os.path.dirname(_found[0])
elif os.path.exists('/content/drive/MyDrive'):
    base_drive_root = '/content/drive/MyDrive/LemGendaryModels'
    drive_target_dir = os.path.join(base_drive_root, model_key, 'checkpoints')
    os.makedirs(drive_target_dir, exist_ok=True)
elif os.path.exists('/content/drive'):
    drive_target_dir = f'/content/drive/MyDrive/LemGendaryModels/{model_key}/checkpoints'
    os.makedirs(drive_target_dir, exist_ok=True)

def drive_sync_worker():
    print(f'[SYNC] Background sync thread started. Target: {drive_target_dir}')
    last_error = None
    while True:
        try:
            for f in os.listdir(ckpt_hub_dir):
                src = os.path.join(ckpt_hub_dir, f)
                if os.path.isfile(src):
                    dst = os.path.join(drive_target_dir, f)
                    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                        tmp_dst = dst + '.tmp'
                        shutil.copy2(src, tmp_dst)
                        os.rename(tmp_dst, dst)
            m_src = os.path.join(model_hub_dir, 'metrics.csv')
            if os.path.exists(m_src):
                m_dst = os.path.join(os.path.dirname(drive_target_dir), 'metrics.csv')
                if not os.path.exists(m_dst) or os.path.getmtime(m_src) > os.path.getmtime(m_dst):
                    shutil.copy2(m_src, m_dst)
            last_error = None
        except OSError as sync_err:
            # Drive unmounted, quota full or a checkpoint mid-write (shutil.Error is an
            # OSError): report each new error once, retry on the next pass.
            if str(sync_err) != last_error:
                last_error = str(sync_err)
                print(f'[SYNC] [WARNING] Drive sync pass failed, retrying in 30 s: '
                      f'{type(sync_err).__name__}: {sync_err}')
        time.sleep(30)

if drive_target_dir:
    t = threading.Thread(target=drive_sync_worker, daemon=True)
    t.start()
else:
    print('[WARNING] No Google Drive checkpoint directory found. Background sync disabled.')


## 10. Nuclear Training Matrix


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

suite_candidates = ['/content/lemgendary-training-suite']
active_suite_dir = next(
    (p for p in suite_candidates if os.path.exists(os.path.join(p, 'training', 'train.py'))),
    '/content/lemgendary-training-suite'
)
if active_suite_dir not in sys.path:
    sys.path.insert(0, active_suite_dir)

os.chdir(active_suite_dir)
print(f'[OK] [SUITE] Active working directory set to: {os.getcwd()}')

try:
    current_pid = os.getpid()
    ps_out = subprocess.check_output(['ps', '-ef'], text=True)
    for line in ps_out.split('\n'):
        if 'train.py' in line and str(current_pid) not in line:
            parts = line.split()
            if len(parts) > 1:
                pid = int(parts[1])
                print(f'[JANITOR] Killing stale zombie training process (PID {pid})...')
                subprocess.run(['kill', '-9', str(pid)], check=False, capture_output=True)
except (OSError, ValueError, subprocess.SubprocessError) as janitor_err:
    print(f'[JANITOR] Stale process scan skipped: {janitor_err}')

print('[LAUNCH] [NUCLEAR] Initiating Training Matrix for ai_helper_lemgend...')
is_forex = False
model_key = 'ai_helper_lemgend'
preferred_parallel = 'single'
# GPUs are counted in a fresh interpreter: torch in this kernel process may be older
# than the build the dependency cell installed.
gpu_probe = subprocess.run(
    [sys.executable, '-c',
     'import torch; print(torch.cuda.device_count() if torch.cuda.is_available() else 0)'],
    check=False, capture_output=True, text=True)
if gpu_probe.returncode == 0 and gpu_probe.stdout.strip():
    num_gpus = int(gpu_probe.stdout.split()[-1])
else:
    num_gpus = 0
    print(f'[WARNING] GPU count probe failed with exit code {gpu_probe.returncode}: '
          f'{gpu_probe.stderr[-1000:]}')
use_ddp = (
    num_gpus > 1
    and (preferred_parallel == 'ddp' or (
        preferred_parallel != 'single' and not is_forex and 'yolo' not in model_key.lower()
    ))
)

if is_forex:
    cmd = [sys.executable, '-u', '-m', 'training.train_forex_curriculum']
elif (os.environ.get('PJRT_DEVICE', '').upper() == 'TPU'
      or os.environ.get('TPU_NAME')
      or os.environ.get('KAGGLE_TPU_NAME')):
    print('[TPU] Google Cloud TPU XLA runtime detected. Launching TPU training...')
    cmd = [
        sys.executable, '-u', 'training/train.py',
        '--model', 'ai_helper_lemgend', '--env', 'colab', '--auto_sync'
    ]
elif use_ddp and os.path.exists('tools/launch_ddp.py'):
    print(f"[DDP] Dual-GPU acceleration detected ({num_gpus} GPUs). "
          "Launching multi-process DDP via tools/launch_ddp.py...")
    cmd = [
        sys.executable, '-u', 'tools/launch_ddp.py',
        '--model', 'ai_helper_lemgend', '--env', 'colab', '--auto_sync'
    ]
else:
    cmd = [
        sys.executable, '-u', 'training/train.py',
        '--model', 'ai_helper_lemgend', '--env', 'colab', '--auto_sync'
    ]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
try:
    import io
    with io.TextIOWrapper(p.stdout, newline='') as trainer_out:
        for line in trainer_out:
            print(line, end='', flush=True)
    p.wait()
except KeyboardInterrupt:
    print('\n[TERMINATED] Training interrupted by user. Terminating training subprocess safely...')
    try:
        p.terminate()
        p.wait(timeout=5)
    except subprocess.TimeoutExpired:
        p.kill()
    print('[OK] Subprocess successfully killed. VRAM and CPU are clean.')

trainer_exit = __import__(
    'training.cloud.trainer_exit', fromlist=['TrainerOutcome', 'check_trainer_exit']
)
check_trainer_exit = trainer_exit.check_trainer_exit
TrainerOutcome = trainer_exit.TrainerOutcome
trainer_outcome = check_trainer_exit(p.wait(), Path(
    '/content/LemGendaryModels/ai_helper_lemgend'
))
if trainer_outcome is TrainerOutcome.DEADLINE_STOP:
    print('[DEADLINE] Training stopped at the session deadline; progress is synced and the next session resumes it.')
else:
    print('[OK] Training finished with exit code 0.')


## 11. SOTA Models Hub Publishing (4 Variations)


In [ ]:
import os
import subprocess
import sys

model_key = 'ai_helper_lemgend'
local_path = '/content/LemGendaryModels/ai_helper_lemgend'
suite_dir = '/content/lemgendary-training-suite'

def _run_suite(module, *args):
    # Suite code that may load torch runs in a fresh interpreter, never in this kernel
    # process: its torch may be older than the build the dependency cell installed.
    with subprocess.Popen([sys.executable, '-u', '-m', module, *args], cwd=suite_dir,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
    return proc.returncode

if os.path.exists('/content/LemGendaryModels/ai_helper_lemgend/session_deadline.json'):
    print('[DEADLINE] Session deadline stop: progress is on the Hub; release publish skipped until training finishes.')
else:
    print(f'[PUBLISH] Exporting and publishing {model_key} to the Kaggle Models Hub...')
    publish_code = _run_suite('training.cloud.release_publish_cli', model_key)
    if publish_code != 0:
        raise RuntimeError(
        f'[ERROR] Release publish of ai_helper_lemgend failed with exit '
        f'code {publish_code}; see the log above.'
    )
